# Pathway Agent — Full Project Explainer

**A guided, beginner-friendly deep dive into every part of this codebase.**

This notebook is a **standalone learning + reference document**. It does not modify, run, or depend on
the original project services — it is safe to keep next to the code forever and re-read as you work.
A few code cells contain small, self-contained *toy demonstrations* (using only `numpy`/`pandas`/`sklearn`/
`matplotlib`, which are almost always available) that reproduce the **mechanism** used by the real services,
so you can see the algorithm work on tiny fake data before you ever touch the real (large, missing-from-git)
models.

> **How to use this notebook:** Read top to bottom once. Then use it as a reference — jump to the section
> for whichever service/file you're currently editing. Every section names the exact file(s) in the repo
> it corresponds to, so you can go read the real source side-by-side.

---

## 0. TL;DR — What is this project?

**"Pathway Agent"** is a Final Year Project (FYP) that builds an **AI research assistant for synthetic
biology / metabolic engineering**. A synthetic biologist wants to engineer a microorganism (e.g. *E. coli*)
to produce a target chemical (e.g. succinate, isobutanol). Answering "can I build this pathway, and will it
work?" normally requires manually consulting several different specialist tools and databases:

- Is a candidate biochemical reaction **thermodynamically favorable** (does it release or require energy)?
- Will a given **enzyme** actually accept a given **substrate** (chemical) well enough to catalyze the reaction?
- Has anyone already found a **known metabolic pathway** and yield for this product in a similar organism?
- What is the official identity of a chemical/reaction/protein ID (KEGG, Rhea, UniProt)?

This project wraps a **Large Language Model (LLM) acting as an autonomous agent** around four specialist
back-end tools (three of them powered by machine learning / scientific computing, one a curated database
search) so that a scientist can just **ask in plain English** and the agent decides which tool(s) to call,
in which order, and stitches the results into a coherent answer — while showing its work.

This is a **multi-service (microservice) system**, not a single script. Each "brain" (the LLM reasoning
loop, and each scientific tool) runs as its own independent web server (using FastAPI), and they talk to
each other over HTTP. A Streamlit web page is the human-facing chat window on top.


## 0.1 Final goal of the project (as stated by the authors)

From the project's own `README.md`:

> *"Pathway Agent is an intelligent assistant designed for strain and pathway engineering in synthetic
> biology. Built on LLM reasoning via LangGraph, it integrates specialized microservices to automate
> pathway design, enzyme ranking, and feasibility analysis."*

Concretely, the deliverable is a working, deployable system where a user can type things like:

- *"Calculate the Gibbs free energy for the reaction: C01083 + C00001 <=> 2 C00031"*
- *"Rank the affinity between enzyme sequence MTKRV...FDN and substrate C00149"*
- *"What is the maximum theoretical yield of succinate in host model iML1515?"*

...and get back a scientifically grounded answer, with the agent's tool calls and raw outputs visible for
verification (important in science — you don't just trust an LLM's word, you can audit what data it used).

There is also a formal **evaluation harness** (LLM-as-a-judge scoring of agent transcripts) used to
measure and report on the agent's quality — this is presumably a core piece of the FYP's academic
contribution (see Section 14).


---
## 1. The Big Picture: System Architecture

The system is composed of **6 independent services**. Every one of them is a small, independently
runnable **web server**. None of them import each other's Python code directly — they only ever
communicate over **HTTP** (i.e. one service sends a JSON request to another service's URL and gets a
JSON response back). This is what "microservices architecture" means in practice.

| # | Service | Port | What it does | Type of logic |
|---|---------|------|---------------|----------------|
| 1 | **frontend** | 8501 | The chat web page a human uses | Streamlit UI, no ML |
| 2 | **agent-core** | 8081 | The "brain": runs the LLM + decision loop, calls the other 4 tools | LangGraph + local LLM (Ollama) |
| 3 | **dGPredictor** | 8002 | Predicts Gibbs free energy (ΔG) of a reaction from chemical structure | Bayesian Ridge Regression (classic ML, *not* deep learning) on hand-engineered molecular features |
| 4 | **EnzRank** | 8003 | Scores how well an enzyme (protein sequence) matches a substrate (chemical) | Convolutional Neural Network (deep learning), GPU-capable |
| 5 | **SearchMEResource** | 8004 | Looks up known, published metabolic engineering pathways/yields | Pure data search (pandas over a cached literature dataset) — **no ML** |
| 6 | **eQuilibrator** | 8005 | Textbook-standard ΔG calculation for *known* biochemical reactions | Third-party library (`equilibrator-api`), Component Contribution method |

Two more supporting pieces are **not services**, but are essential:

- **Ollama** — a local program (installed separately, not part of this repo) that runs the actual LLM
  (`qwen2.5:7b`) on the machine's CPU/GPU. `agent-core` just makes HTTP calls to Ollama, exactly like it
  calls the other tool services. No API keys or internet calls to OpenAI/Anthropic are used — the LLM
  runs 100% locally.
- **Apptainer / Singularity containers** — like Docker, but designed for shared university/HPC servers
  where users don't have root access. Each service is packaged into a `.sif` image file and launched with
  `apptainer exec`. See Section 12.


In [ ]:

# A quick visual map of the architecture, drawn with plain matplotlib.
# This is purely illustrative (no dependency beyond matplotlib) - reuse/embed this diagram in your
# reports to your supervisor/boss if useful.

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

def box(x, y, w, h, text, color):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05",
                                    linewidth=1.5, edgecolor='black', facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=9, wrap=True)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=15,
                                  color='gray', linewidth=1.2))

# User / Frontend
box(3.7, 8.7, 2.6, 0.9, "User\n(browser)", "#e8f0fe")
box(3.7, 7.2, 2.6, 0.9, "frontend :8501\n(Streamlit chat UI)", "#cfe2f3")
arrow(5, 8.7, 5, 8.1)

# Agent core
box(3.4, 5.5, 3.2, 1.0, "agent-core :8081\nLangGraph orchestrator\n(FastAPI)", "#fce5cd")
arrow(5, 7.2, 5, 6.5)

# Ollama
box(0.3, 5.6, 2.4, 0.8, "Ollama\n(local LLM: qwen2.5:7b)", "#f4cccc")
arrow(3.4, 6.0, 2.7, 6.0)
ax.annotate('', xy=(3.4, 5.9), xytext=(2.7, 5.9))

# Tools row
tool_y = 3.5
box(0.2, tool_y, 2.1, 1.1, "dGPredictor\n:8002\nBayesianRidge\n(ML: dG predict)", "#d9ead3")
box(2.5, tool_y, 2.1, 1.1, "EnzRank\n:8003\nCNN\n(ML: enzyme score)", "#d9ead3")
box(4.8, tool_y, 2.1, 1.1, "SearchMEResource\n:8004\npandas lookup\n(no ML)", "#fff2cc")
box(7.1, tool_y, 2.1, 1.1, "eQuilibrator\n:8005\nComponent\nContribution", "#d9ead3")

for x in [1.25, 3.55, 5.85, 8.15]:
    arrow(x, 5.5, x, tool_y + 1.1)

# External APIs
box(3.4, 1.2, 3.2, 0.9, "External REST APIs:\nKEGG / Rhea / UniProt\n(entity_search_tool,\nentity_validation_tool)", "#ead1dc")
arrow(5, tool_y, 5, 2.1)

ax.set_title("Pathway Agent — service map (arrows = HTTP calls)", fontsize=12)
plt.tight_layout()
plt.show()


### Reading the diagram

1. The user types a question into the **frontend** (Streamlit page).
2. The frontend does one `requests.post()` to **agent-core**'s `/chat` endpoint with the raw text.
3. **agent-core** feeds the message into a **LangGraph state machine** that repeatedly asks the local LLM
   (via **Ollama**) "what should happen next?". The LLM can either answer directly, or ask to call one of
   6 registered "tools".
4. Each tool is a thin Python wrapper (`services/agent-core/app/tools/*.py`) that makes another HTTP call —
   either to one of the 4 specialist microservices, or directly to a public bioinformatics REST API
   (KEGG/Rhea/UniProt) for looking up/validating chemical & protein identifiers.
5. Results flow back up through agent-core, get turned into natural language by the LLM, and are sent back
   to the frontend, which renders the chat bubble (plus an expandable "tool execution log" for
   transparency/debugging).

This is the **"agentic" / "ReAct" pattern**: LLM → decide action → take action → observe result → LLM
decides again → ... → final answer. Section 5 explains exactly how this loop is implemented in code.


---
## 2. Repository Layout

```
FYP2026xintong-main/
├── README.md                      <- top-level deployment & usage instructions
├── deploy_dev.sh                  <- the single script that launches ALL 6 services
├── missing_models_and_data.txt    <- list of ~10,700 files that exist on the real deployment
│                                       server but are NOT checked into this git copy (see Sec. 13)
│
└── services/
    ├── frontend/                  <- Streamlit chat UI (port 8501)
    │   ├── app.py                     the entire UI logic (single file)
    │   ├── md/                        markdown text shown in the sidebar "Knowledge Base" buttons
    │   ├── requirements.txt
    │   └── Dockerfile
    │
    ├── agent-core/                <- the LLM orchestrator (port 8081)
    │   ├── main.py                     FastAPI app; exposes /chat and /history/{id}
    │   ├── app/
    │   │   ├── config.py               all service URLs + LLM model name, read from env vars
    │   │   ├── agents/graph.py         *** THE CORE FILE *** — LangGraph state machine, system prompt
    │   │   └── tools/                  one file per tool the agent can call
    │   │       ├── base.py                 shared `post_request()` HTTP helper
    │   │       ├── remote_dg.py             wraps dGPredictor service
    │   │       ├── remote_enz.py            wraps EnzRank service
    │   │       ├── remote_me.py             wraps SearchMEResource service
    │   │       ├── remote_equilibrator.py   wraps eQuilibrator service
    │   │       └── db_search.py             calls public KEGG/Rhea/UniProt REST APIs directly
    │   ├── evaluation/                 automated test harness + LLM-as-judge scoring (Sec. 14)
    │   ├── requirements.txt
    │   └── Dockerfile / agent.def
    │
    ├── dGPredictor/                <- ΔG (Gibbs free energy) prediction ML service (port 8002)
    │   ├── server.py                   FastAPI app; writes a temp JSON, shells out to batch_predict.py
    │   ├── batch_predict.py            loads the pretrained regression model + runs predictions
    │   ├── db_bulk_dg_gen.py           the actual feature-engineering + prediction functions
    │   ├── decompose_groups.py         RDKit molecular fragment ("group") decomposition
    │   ├── model_gen.py                OFFLINE training script that PRODUCED the .pkl model file
    │   ├── predict.py                  older/alternate CLI experimentation script (legacy)
    │   ├── CC/                         "Component Contribution" helper module (compound cache, thermo constants)
    │   ├── streamlit/, analysis_*.ipynb, retrieve_bulk*.ipynb  <- exploratory/legacy notebooks
    │   ├── figures/                    plots from the original research (regression fit quality etc.)
    │   ├── requirements.txt
    │   └── Dockerfile
    │
    ├── EnzRank/                    <- enzyme-substrate compatibility CNN service (port 8003)
    │   ├── server.py                   FastAPI app, loads the Keras model once at startup
    │   ├── batch_predict.py            EnzRankService class: feature encoding + model.predict()
    │   ├── app.py / Streamlit/main.py  legacy standalone Streamlit demo app (same logic, older style)
    │   ├── requirements.txt (TensorFlow/Keras pinned versions)
    │   └── Dockerfile
    │
    ├── SearchMEResource/           <- literature pathway/yield lookup service (port 8004)
    │   ├── main.py                     FastAPI app; exposes /query
    │   ├── me_agent_logic.py           MEResourceAgent class: loads Excel/TSV data, ranks candidate pathways
    │   ├── requirements.txt
    │   └── search_me.def
    │
    └── eQuilibrator/               <- standard thermodynamics microservice (port 8005)
        ├── main.py                     tiny FastAPI wrapper around the `equilibrator_api` pip package
        ├── requirements.txt
        └── equilibrator.def
```

Two directories referenced in the README (`data/agent_memory/`, `images/`, `logs/`) are **created at
deploy time** by `deploy_dev.sh` — they don't ship in git because they hold runtime state (SQLite DB,
built container images, log files).


In [ ]:

# Live check: walk the actual repo on THIS machine and print what is really present,
# so you can see at a glance what matches the map above right now.
from pathlib import Path

# Adjust this if you run the notebook from somewhere other than the repo root.
ROOT = Path.cwd()
if not (ROOT / "services").exists():
    # try the directory this notebook file lives in
    candidate = Path("/Users/yiranzhu/Desktop/python/FYP2026xintong-main")
    if candidate.exists():
        ROOT = candidate

print(f"Using project root: {ROOT}\n")

if (ROOT / "services").exists():
    for service_dir in sorted((ROOT / "services").iterdir()):
        if service_dir.is_dir():
            n_files = sum(1 for _ in service_dir.rglob("*") if _.is_file())
            print(f"services/{service_dir.name:<20} -> {n_files:4d} files present")
else:
    print("Could not locate the 'services/' directory automatically — "
          "set ROOT manually to the project path.")


---
## 3. Full Technology / Library Stack

### 3.1 Shared web framework (used by every backend service)

| Library | Role |
|---|---|
| **FastAPI** | Defines each service's HTTP API (routes like `/chat`, `/predict`, `/rank`, `/query`, `/calculate`). Modern, type-checked, auto-generates OpenAPI docs. |
| **uvicorn** | The actual ASGI web server process that runs a FastAPI app. |
| **pydantic** | Defines the *shape* of request/response JSON bodies as Python classes (`BaseModel`), with automatic validation. |
| **requests** | Simple synchronous HTTP client, used for service-to-service calls. |

### 3.2 `agent-core` — the LLM / agent stack

| Library | Version pin | Role |
|---|---|---|
| **langchain-ollama** | 0.1.2 | Adapter that lets LangChain/LangGraph talk to a locally-running Ollama LLM server. |
| **langchain-core** | 0.2.36 | Base abstractions: `HumanMessage`, `SystemMessage`, the `@tool` decorator, etc. |
| **langchain** / **langchain-community** | 0.2.14 / 0.2.12 | Higher-level LangChain glue (installed as dependencies; graph.py mostly uses langchain-core + langgraph directly). |
| **langgraph** | ≥0.2.0 | The state-machine / agent-loop engine — the real "brain" of the orchestration. See Section 5. |
| **langgraph-checkpoint-sqlite**, **aiosqlite** | — | Persists the agent's running conversation state to a SQLite file so chats survive restarts (long-term memory). |
| **Ollama** (external program, not a Python package) | — | Runs the LLM itself (`qwen2.5:7b` by default) locally, exposing an HTTP API on port 11434. |

### 3.3 `dGPredictor` — chemistry + classic ML stack

| Library | Role |
|---|---|
| **RDKit** | Cheminformatics toolkit: parses SMILES strings into molecule graphs, generates circular substructure fragments and Morgan fingerprints. |
| **scikit-learn** | `BayesianRidge`, `LinearRegression`, `RidgeCV` — the regression models that actually predict ΔG. |
| **pandas / NumPy** | Data wrangling, feature vector construction. |
| **scipy** (`loadmat`/`savemat`) | Reads `.mat` (MATLAB) files — the original "Component Contribution" research data was produced in MATLAB. |
| **joblib / pickle** | Serializes/deserializes the trained regression model to/from disk (`model/M12_model_BR.pkl`). |
| **PuLP** | A linear-programming solver, used by `mini_novoStoic.py` for pathway design optimization (not on the hot request path of the FastAPI service). |
| **OpenBabel** | Alternative cheminformatics toolkit, used for some format conversions. |
| **Streamlit / matplotlib** | Used only by the legacy standalone demo app, not the production FastAPI server. |

### 3.4 `EnzRank` — deep learning stack

| Library | Version pin | Role |
|---|---|---|
| **TensorFlow / Keras** | 2.10.0 | Loads and runs the pretrained Convolutional Neural Network (`Final_model.model`). |
| **RDKit** | — | Generates the 2048-bit Morgan fingerprint for the substrate molecule. |
| **h5py** | 3.7.0 | Reads the Keras model's saved weights file format. |
| **pandas / scikit-learn / scipy** | — | Data loading (KEGG compound CSV) and general utility. |

### 3.5 `eQuilibrator` — standard thermodynamics library

| Library | Role |
|---|---|
| **equilibrator-api** | Third-party package implementing the peer-reviewed *Component Contribution* method with a bundled precomputed thermodynamic database (~1.5 GB), pH/ionic-strength correction. |
| **pint** | Physical-units handling (e.g. `Q_(7.3)` for pH, kJ/mol for energy) so values can't be silently misinterpreted in the wrong unit. |
| **SQLAlchemy, networkx, scipy** | Internal dependencies of `equilibrator-api` (database access + graph algorithms for compound relationships). |

### 3.6 `SearchMEResource` — data engineering stack (no ML)

| Library | Role |
|---|---|
| **pandas** | Loading and filtering the metabolic-engineering literature dataset. |
| **openpyxl** | Reads the original `.xlsx` supplementary-material spreadsheets from a published Nature paper. |
| **pyarrow** | Reads/writes the `.parquet` cache file (a fast on-disk column format) built from those spreadsheets on first run. |

### 3.7 `frontend` — UI stack

| Library | Role |
|---|---|
| **Streamlit** | Turns a plain Python script into an interactive web app — no HTML/JS/CSS needed (a little custom CSS is injected for styling). |
| **requests** | Calls the agent-core `/chat` HTTP endpoint. |

### 3.8 Infrastructure / deployment (not Python libraries)

| Tool | Role |
|---|---|
| **Apptainer (Singularity)** | Container runtime used instead of Docker — common on shared university/HPC Linux servers without root access. Each service ships a `.def` (definition) file describing how to build its `.sif` image. |
| **Docker** | `Dockerfile`s also exist for several services (dual packaging — can run via Docker OR Apptainer). |
| **Ollama** | Local LLM inference server. |
| **bash** (`deploy_dev.sh`) | Orchestrates starting/stopping all 6 services with the right environment variables and log redirection. |


---
## 4. Glossary for Beginners (read this before Section 5+)

You said you have beginner-level ML knowledge — here are the concepts this codebase actually uses,
explained plainly, in the order you'll meet them below.

| Term | Plain explanation |
|---|---|
| **LLM (Large Language Model)** | A neural network trained on huge amounts of text that predicts "what word comes next", which lets it hold conversations, follow instructions, and (with the right scaffolding) decide to call external tools. Here it's `qwen2.5:7b`, run locally by **Ollama** (no cloud API). |
| **Agent** | An LLM wrapped in a loop that lets it take *actions* (like calling a function/tool), observe the result, and decide what to do next — repeatedly — instead of just producing one static text reply. |
| **Tool calling / function calling** | A protocol where the LLM outputs a structured request like `{"name": "dg_predictor_tool", "args": {...}}` instead of plain prose, and the surrounding code intercepts that, actually runs the corresponding Python function, and feeds the result back to the LLM. |
| **LangGraph** | A Python library for building agents as an explicit **graph of nodes and edges** (a state machine) rather than a hidden loop — you can see and control exactly when the LLM is called vs. when a tool is called. See Section 5. |
| **State machine** | A system that is always in exactly one "state" and moves to another state based on defined rules/edges. Here the states are informally "agent is thinking" and "a tool is executing". |
| **Checkpointing (SqliteSaver)** | Saving the full conversation state to disk (SQLite) after every step, keyed by a `thread_id`, so a conversation can be resumed later or inspected — this is the agent's "memory". |
| **Regression** | A statistics/ML technique that fits a mathematical function to predict a **continuous number** (e.g. −250.3 kJ/mol) from input features — as opposed to *classification*, which predicts a category. |
| **Linear Regression** | The simplest regression: predicted value = weighted sum of input features (`y = w1*x1 + w2*x2 + ...`). |
| **Ridge Regression** | Linear regression with an added penalty that discourages huge weights, which helps when you have many correlated/noisy features (common in chemistry data) — reduces overfitting. |
| **Bayesian Ridge Regression** | Like Ridge Regression, but formulated probabilistically — instead of a single "best" prediction, it naturally also gives you an **uncertainty estimate (a standard deviation)** for each prediction. This is exactly why dGPredictor can report a ΔG value *and* an error bar. |
| **Feature vector** | A list of numbers that represents an input (a molecule, a protein, a reaction) in a form a model can do math on. Constructing good feature vectors from raw chemistry/biology data is most of the "engineering" work in this repo. |
| **Molecular fingerprint (Morgan / ECFP)** | A standard cheminformatics technique: walk outward from every atom in a molecule up to a fixed "radius" of neighboring bonds, hash each resulting sub-structure into one of e.g. 2048 bit positions, and set that bit to 1. Two molecules with similar structure end up with similar bit patterns — this is how a computer "sees" chemical structure as a fixed-length number vector. |
| **Group contribution method** | A classical (pre-deep-learning) approach in physical chemistry: break a molecule/reaction into known structural "groups" (functional groups, ring fragments...), and assume the total thermodynamic property (like ΔG) is a **linear combination** of each group's individually-known contribution. dGPredictor's regression is trained to *learn* these per-group contributions from data. |
| **CNN (Convolutional Neural Network)** | A neural network architecture that applies small sliding filters over its input to detect local patterns, originally for images, but also effective on sequences (protein sequences here) and on fingerprint-like vectors. EnzRank uses one, but its exact architecture/training code is not present in this git copy — see Section 8.4. |
| **Embedding / sequence encoding** | Converting each amino acid letter (A, I, L, V, ...) into an integer, so a sequence like `"MTKR..."` becomes a list of numbers a neural network can process. |
| **Padding** | Neural networks usually need fixed-length input. Protein sequences vary in length, so short ones are padded with zeros (and here, sequences are capped/padded to length 2500). |
| **Gibbs free energy (ΔG)** | A number from thermodynamics (kJ/mol) that tells you whether a chemical reaction will proceed spontaneously (negative ΔG, favorable) or requires energy input (positive ΔG, unfavorable) — the single most important feasibility check for a proposed metabolic pathway. |
| **KEGG / Rhea / UniProt IDs** | Public standardized identifiers: KEGG (`C00002`, `R00001`) for compounds/reactions, UniProt (`P12345`) for proteins, Rhea for reactions. The agent constantly needs to translate between human names ("ATP") and these formal IDs before calling the ML tools. |
| **LLM-as-a-judge** | An evaluation technique where you use a *second* LLM to read a transcript and score how good the *first* LLM's answer was (e.g. "rate accuracy 1–5"), because writing exact-match automated tests for open-ended natural language answers is impractical. Used in this project's `evaluation/` folder. |


---
## 5. Deep Dive #1 — `agent-core`: The LangGraph Orchestrator

**Files:** `services/agent-core/app/agents/graph.py` (the core logic), `services/agent-core/main.py`
(the thin FastAPI wrapper), `services/agent-core/app/config.py` (all URLs/model name).

This is the single most important file in the whole repository to understand deeply, because it defines
*how the agent thinks*.

### 5.1 The State

```python
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
```

Everything the graph tracks is just **one growing list of chat messages** (`HumanMessage`, `AIMessage`,
`ToolMessage`, `SystemMessage`). `add_messages` is a special "reducer" function LangGraph uses so that
each node can *return only the new message(s) it produced*, and the framework automatically appends them
to the running list rather than you having to manually manage the whole list yourself.

### 5.2 The Nodes

The graph has exactly two "processing nodes":

1. **`"agent"` node → the `chatbot()` function.** Takes the last (up to) 10 messages, prepends the giant
   `SYSTEM_PROMPT` if one isn't already present, and calls `llm_with_tools.invoke(...)`. The LLM either
   returns plain text (a final answer) **or** returns a message with a populated `tool_calls` field (a
   request to run a specific tool with specific arguments).
2. **`"tools"` node → LangGraph's built-in `ToolNode(tools)`.** Looks at the last message's `tool_calls`,
   actually executes the matching Python function(s) from the `tools` list, and turns each result into a
   `ToolMessage` appended back onto the state.

### 5.3 The Edges (the actual "loop")

```python
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)   # "agent" -> "tools" OR "agent" -> END
workflow.add_edge("tools", "agent")                          # "tools" ALWAYS goes back to "agent"
```

This wiring is exactly the classic **ReAct agent loop**:

```
START -> agent -> (does it want a tool?) --yes--> tools -> agent -> (again) -> ... -> END
                          |
                          no
                          v
                         END
```

`should_continue()` is the routing function that decides `"tools"` vs. `END`. Its primary check is simple:
"does the LLM's last message have `tool_calls` populated?" (this is the framework's standard, structured
tool-calling mechanism). But notice there is also a **fallback branch**: if `qwen2.5:7b` fails to use
proper structured tool calling and instead just writes something like `{"name": "dg_predictor_tool", ...}`
as plain text, the code manually detects JSON-looking text containing known tool-name keywords, parses
it by hand, and force-constructs a `tool_calls` entry itself. **This is a workaround for a known weakness
of smaller/local LLMs** — worth remembering if you ever swap in a different Ollama model and things stop
working: check if this fallback parser's keyword list needs updating.

### 5.4 The System Prompt — a hand-engineered decision policy

`SYSTEM_PROMPT` is not a generic "you are a helpful assistant" string — it's a fairly detailed **rulebook**
that exists because a 7B local model isn't reliable enough to figure out subtle domain rules on its own.
Key rules it hard-codes:

- **Never use LaTeX** in output (smaller models like to output `\Delta G` — the prompt explicitly bans it
  because the chat UI renders plain markdown, not LaTeX).
- **Two different ΔG tools exist and must not be confused**: `equilibrator_tool` (for *known* KEGG
  reactions, IDs need a `kegg:` prefix) vs. `dg_predictor_tool` (for *novel/synthetic* reactions, IDs must
  NOT have a prefix). This id-prefix distinction is a very easy mistake for an LLM to make, so it's
  spelled out multiple times in the prompt.
- Four **named workflow scenarios** (IDs given / names given / pathway design requested / full pipeline)
  each with a step-by-step procedure the model should follow, e.g. always call `entity_validation_tool`
  first to double check an ID actually means what the user thinks it means before running an expensive
  calculation on it.

This is a good example of **prompt engineering as a substitute for fine-tuning**: instead of training a
custom model, the developers wrote a very explicit prompt to constrain a general-purpose local LLM's
behavior to this narrow domain.

### 5.5 Memory / persistence

```python
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
memory = SqliteSaver(conn)
...
app_graph = workflow.compile(checkpointer=memory)
```

Every invocation is tagged with a `thread_id` (the frontend uses the Streamlit session's UUID). LangGraph
automatically saves the full state (message history) to the SQLite file after each step, keyed by that
`thread_id`. This is why re-opening a previous chat in the sidebar shows its full history — it's read
straight back out of the database via `get_history_from_db()`, not held in server RAM.

### 5.6 `run_graph()` — what the FastAPI endpoint actually calls

1. Look up how many messages already existed for this `thread_id` (`len_before`).
2. Invoke the compiled graph with the new `HumanMessage`.
3. The graph runs the loop above until `should_continue` returns `END`.
4. Slice out just the *new* messages (everything after `len_before`) to build a human-readable
   `logs` list — this is exactly what populates the "View Tool Execution Logs (Internal)" expander
   in the Streamlit UI.
5. Return `(final_response_text, logs)`.


In [ ]:

# ------------------------------------------------------------------
# TOY DEMO: a from-scratch, dependency-free simulation of the exact
# agent-loop pattern used in graph.py (no LangGraph, no real LLM —
# just plain Python), so you can see the control flow with your own
# eyes and step through it with a debugger if you like.
# ------------------------------------------------------------------

def fake_tool_dg_predictor(reaction_equation):
    '''Pretend version of dg_predictor_tool.'''
    return {"dG": -12.3, "std": 1.1}

def fake_tool_enzrank(enzyme, substrate):
    '''Pretend version of enzrank_tool.'''
    return {"enzrank_score": 0.87}

TOOLS = {
    "dg_predictor_tool": fake_tool_dg_predictor,
    "enzrank_tool": fake_tool_enzrank,
}

def fake_llm_step(messages):
    '''
    Stand-in for `llm_with_tools.invoke(messages)`.
    A REAL LLM decides this dynamically from the conversation;
    here we just hard-code a 2-step scripted "reasoning" for demonstration.
    '''
    n_tool_results = sum(1 for m in messages if m["role"] == "tool")
    if n_tool_results == 0:
        # Step 1: the "LLM" decides it needs a tool
        return {"role": "ai", "tool_call": {"name": "dg_predictor_tool",
                                             "args": {"reaction_equation": "C00002 + C00001 <=> C00008"}}}
    else:
        # Step 2: the "LLM" has the tool result and writes a final answer
        last_tool_result = messages[-1]["content"]
        return {"role": "ai", "content": f"The reaction is favorable, dG = {last_tool_result['dG']} kJ/mol."}

def should_continue(last_message):
    return "tools" if "tool_call" in last_message else "END"

def run_agent_loop(user_query, max_steps=5):
    messages = [{"role": "human", "content": user_query}]
    for step in range(max_steps):
        ai_message = fake_llm_step(messages)
        messages.append(ai_message)
        decision = should_continue(ai_message)
        print(f"[step {step}] agent node ran -> routing decision: {decision}")

        if decision == "END":
            return ai_message["content"], messages

        # "tools" node: actually execute the requested tool
        call = ai_message["tool_call"]
        result = TOOLS[call["name"]](**call["args"]) if call["name"] != "enzrank_tool" else \
                 TOOLS[call["name"]](enzyme="X", substrate="Y")
        messages.append({"role": "tool", "name": call["name"], "content": result})
        print(f"           tool node ran '{call['name']}' -> {result}")

    return "Max steps reached", messages

final_answer, full_transcript = run_agent_loop("Calculate dG for C00002 + C00001 <=> C00008")
print("\nFINAL ANSWER:", final_answer)


Notice the printed trace matches the real `START -> agent -> tools -> agent -> END` pattern exactly —
the only difference in the real system is that `fake_llm_step` is replaced by an actual call to a 7-billion
parameter language model, and `should_continue` has the extra manual-JSON-parsing fallback described above.


---
## 6. Deep Dive #2 — The Agent's 6 Tools

**Files:** everything in `services/agent-core/app/tools/`.

Each tool is a plain Python function decorated with LangChain's `@tool(...)`, with a `pydantic` model
declaring its expected arguments (this schema is what actually gets shown to the LLM so it knows how to
call the tool correctly — LangChain auto-converts the pydantic schema into the JSON-schema format the LLM's
tool-calling API expects).

| Tool name | File | Calls out to | Purpose |
|---|---|---|---|
| `dg_predictor_tool` | `remote_dg.py` | dGPredictor service (`:8002/predict`) | ΔG for **novel/synthetic** reactions, plain KEGG-style IDs, no prefix. Validates the equation string locally first (checks for a `<=>`/`->`/`=` separator and non-empty both sides) before ever making the network call — cheap fail-fast validation. |
| `equilibrator_tool` | `remote_equilibrator.py` | eQuilibrator service (`:8005/calculate`) | ΔG'° for **known/standard** reactions. Auto-prefixes bare KEGG ids with `kegg:` using a regex, so the LLM doesn't even have to get the prefix rule perfectly right. |
| `enzrank_tool` | `remote_enz.py` | EnzRank service (`:8003/rank`) | Enzyme–substrate compatibility score (0 to 1). Accepts a list of `{enzyme, substrate}` dict pairs (batchable). |
| `search_me_resource_tool` | `remote_me.py` | SearchMEResource service (`:8004/query`) | Looks up literature-known pathways/yields for a target product + host organism + carbon source. |
| `entity_search_tool` | `db_search.py` | Public **KEGG** REST API directly (`rest.kegg.jp`) | Translate a common chemical/reaction **name** into KEGG ID candidates. Cleans charge annotations like `(2-)` and hyphens before searching, to improve hit rate. |
| `entity_validation_tool` | `db_search.py` | Public **KEGG**, **Rhea**, or **UniProt** REST APIs directly | Given an ID, confirm it's real and fetch its official name/formula/equation — used as a sanity check before running expensive calculations on a possibly-wrong ID. |

All of the "remote" tools funnel their actual HTTP call through the shared `post_request()` helper in
`base.py`, which standardizes error handling into readable strings (e.g. `"Connection Error: Cannot reach
service at ..."`), so the LLM always gets *some* informative text back even when a backend service is
down — instead of a raw exception.

### 6.1 Why two different ΔG tools at all?

This is a subtle but important design point worth understanding (and worth explaining to your supervisor):

- **eQuilibrator** wraps a peer-reviewed, precomputed, extremely accurate database. But it can **only**
  answer questions about reactions built entirely from compounds already in its database (real, known
  biochemistry).
- **dGPredictor** is a machine-learning *estimator*. It's less precise (hence it also reports a standard
  deviation / uncertainty), but it can generalize to **brand-new or hypothetical reactions** — exactly the
  case that matters most for *designing new* synthetic pathways, which is the whole point of the project.

So the two tools trade off **accuracy vs. generality**, and the system prompt's job is to make sure the
LLM picks the right one for the right situation.


---
## 7. Deep Dive #3 — `dGPredictor`: Group-Contribution Machine Learning

**Files:** `services/dGPredictor/server.py`, `batch_predict.py`, `db_bulk_dg_gen.py`,
`decompose_groups.py`, `model_gen.py` (offline training), `CC/` (Component Contribution helpers).

### 7.1 The request path (how a prediction actually happens at runtime)

This service has an unusual and important implementation detail: **the FastAPI endpoint does not run the
model in-process.** Look at `server.py`:

```python
cmd = ["python", "batch_predict.py", "--input_file", input_path, "--output_file", output_path]
result = subprocess.run(cmd, check=True, cwd=BASE_DIR, capture_output=True, text=True)
```

The flow is:
1. `/predict` receives JSON like `{"data": {"R_ab12": "C00002 + C00001 <=> C00008"}}`.
2. It writes that dict to a **temporary JSON file** on disk.
3. It **spawns a brand-new Python process** running `batch_predict.py --input_file ... --output_file ...`.
4. That subprocess loads the model + feature data **from scratch**, computes predictions, and writes a
   result JSON file.
5. The FastAPI endpoint reads that output file back in, deletes both temp files in a background task, and
   returns the JSON.

**Why does this matter to you as an engineer?** This is almost certainly a workaround for some
compatibility/legacy-code issue (the underlying scientific code looks like it was written for Python 2 /
older library versions originally — note the `.iteritems()` calls and `print x` statements still present
in some of the older, unused functions in `predict.py` and `db_bulk_dg_gen.py`'s legacy branch). Running it
as a subprocess isolates any crashes/leaks, but it also means **every single request pays the full cost of
reloading the regression model and all reference data files from disk** — there's no persistent model
loaded in memory the way `EnzRank` does it (compare to `EnzRank/server.py`'s `@app.on_event("startup")`
pattern, which loads the model exactly once). If you're asked to improve performance later, converting
`dGPredictor` to load its model once at startup (like EnzRank does) instead of once-per-request via
subprocess is a concrete, well-scoped improvement task.

### 7.2 The science: how a reaction becomes a feature vector

This is the "group contribution method" mentioned in the glossary, concretely:

**Step 1 — Molecular fragmentation (`decompose_groups.py` / `count_substructures()`)**

For every atom in a molecule, RDKit's `FindAtomEnvironmentOfRadiusN(molecule, radius, atom_index)` finds
all bonds within `radius` bonds of that atom, and the corresponding fragment is converted to a canonical
SMILES string ("this specific ring/functional-group shape"). Counting how many times each unique fragment
shape occurs across the whole molecule produces a **dictionary: {fragment_shape: count}**. This is done at
**two different radii (1 and 2)** — radius 1 captures small/local groups, radius 2 captures larger
neighborhoods — and both are used together as "molecular signatures".

**Step 2 — Reaction rule vector (`get_rule()` in `db_bulk_dg_gen.py`)**

A chemical reaction is really just "some amount of each compound consumed, some amount produced"
(negative/positive stoichiometric coefficients). The reaction's feature vector is built by taking the
**stoichiometry-weighted sum of the group vectors of every compound in the reaction**:

```
reaction_vector[fragment] = Σ over compounds (stoichiometric_coefficient × count_of_fragment_in_that_compound)
```

Intuitively: if a fragment is destroyed on the reactant side and newly created on the product side, this
correctly captures "this reaction converts fragment shape A into fragment shape B", which is exactly the
chemistry that determines the reaction's energy change. Water (`C00080`, i.e. H⁺, and `C00282`, H₂) are
explicitly excluded from the group counting (their thermodynamic contribution is handled separately, via
pH-based correction in `get_ddG0()`).

Both radius-1 and radius-2 vectors are concatenated together (with some fixed zero-padding columns kept
for compatibility with the exact shape the original trained model expects) into one long feature vector, `X`.

**Step 3 — Regression (`model_gen.py` offline; `get_dG0()` at runtime)**

```python
regr_rcombined = BayesianRidge(tol=1e-6, fit_intercept=False, compute_score=True).fit(Xrc, y)
...
ymean, ystd = loaded_model.predict(X, return_std=True)
```

The model was **trained once, offline**, on a large table of experimentally-measured reference reactions
(`data/Test_KEGG_all_grp.mat`) and saved to `model/M12_model_BR.pkl` via `joblib.dump()`. At request time,
the already-trained model is just loaded and asked to `.predict()` on the new reaction's feature vector.
`return_std=True` is the special Bayesian-Ridge capability that gives back **both a mean prediction and an
uncertainty (standard deviation)** — this is exactly the `dG` and `std` fields you saw returned by the
`/predict` endpoint and reported to the user as "Std Dev: ...".

Finally, `get_dG0()` adds a **pH/ionic-strength correction term** (`get_ddG0()`, computed from
`CC/thermodynamic_constants.py` and `CC/compound.py`) on top of the raw group-contribution prediction,
because the "standard" ΔG values used in training assume specific reference conditions, and real
biochemical reactions happen at a specific pH (default 7.0) that must be corrected for.

> **Note:** `model_gen.py` is offline/one-time training code — it is *not* run as part of the live service.
> It exists in the repo purely so a future engineer (you!) can see how the shipped `.pkl` model file was
> produced, and could retrain it on new data if needed.


In [ ]:

# ------------------------------------------------------------------
# TOY DEMO: reproduce the ENTIRE dGPredictor mechanism end-to-end on
# a tiny made-up "chemistry", using only numpy + scikit-learn.
# This mirrors get_rule() + BayesianRidge.predict(return_std=True)
# exactly, just with 4 pretend molecular fragments instead of
# thousands of real ones, and 2 pretend compounds instead of KEGG's
# full database.
# ------------------------------------------------------------------
import numpy as np
from sklearn.linear_model import BayesianRidge

# --- "Molecular signatures": {compound_id: {fragment_name: count}} ---
molsig = {
    "C_A": {"frag_ring":   2, "frag_OH": 1, "frag_C=O": 0},
    "C_B": {"frag_ring":   0, "frag_OH": 0, "frag_C=O": 1},
    "C_C": {"frag_ring":   2, "frag_OH": 0, "frag_C=O": 1},   # A's ring survives, OH -> C=O
}
fragment_names = ["frag_ring", "frag_OH", "frag_C=O"]

def reaction_to_feature_vector(rxn_dict, molsig, fragment_names):
    '''This is exactly the logic of get_rule() in db_bulk_dg_gen.py, simplified.'''
    vec = np.zeros(len(fragment_names))
    for compound_id, stoich in rxn_dict.items():
        sig = molsig.get(compound_id, {})
        for i, frag in enumerate(fragment_names):
            vec[i] += stoich * sig.get(frag, 0)
    return vec

# A pretend training set of "known" reactions with measured dG values (kJ/mol)
training_reactions = [
    ({"C_A": -1, "C_C": 1},              -12.4),
    ({"C_A": -1, "C_B": -1, "C_C": 1},   -30.1),
    ({"C_C": -1, "C_A": 1},               12.6),   # reverse of the first
    ({"C_B": -2, "C_C": 1, "C_A": 1},    -18.9),
]

X_train = np.array([reaction_to_feature_vector(rxn, molsig, fragment_names) for rxn, _ in training_reactions])
y_train = np.array([dg for _, dg in training_reactions])

model = BayesianRidge(fit_intercept=False, compute_score=True)
model.fit(X_train, y_train)

print("Learned per-fragment contribution to dG (kJ/mol):")
for name, coef in zip(fragment_names, model.coef_):
    print(f"  {name:12s}: {coef:8.3f}")

# Predict a brand-new, never-seen reaction: C_A -> C_B + C_C  (just an example)
novel_reaction = {"C_A": -1, "C_B": 1, "C_C": 1}
X_new = reaction_to_feature_vector(novel_reaction, molsig, fragment_names).reshape(1, -1)
mean_pred, std_pred = model.predict(X_new, return_std=True)

print(f"\nPredicted dG for novel reaction {novel_reaction}:")
print(f"  mean = {mean_pred[0]:.2f} kJ/mol,  std = {std_pred[0]:.2f} kJ/mol")
print("\n(This is precisely what dGPredictor's /predict endpoint returns for a real reaction —")
print(" just using thousands of real RDKit-derived fragments instead of our 3 pretend ones.)")


**Try it yourself:** change `training_reactions` above (add more, change the dG values) and re-run —
watch how the learned per-fragment contributions and the prediction on `novel_reaction` change. This is
the exact mental model for what happens when the real `model_gen.py` is re-run on updated experimental
data.


---
## 8. Deep Dive #4 — `EnzRank`: Deep Learning Enzyme–Substrate Scoring

**Files:** `services/EnzRank/server.py`, `batch_predict.py` (production inference path),
`app.py` / `Streamlit/main.py` (legacy standalone demo — same logic duplicated in an older style).

### 8.1 What question does it answer?

Given a **protein's amino-acid sequence** (the enzyme) and a **chemical** (the substrate, identified either
by KEGG ID or a raw SMILES string), output a single score between 0 and 1 estimating how well that enzyme
is likely to act on that substrate. This helps a synthetic biologist pick which real-world enzyme to use
to implement a step in a designed pathway.

### 8.2 Runtime architecture (this one loads the model ONCE, correctly)

Unlike dGPredictor, `EnzRank/server.py` uses FastAPI's startup hook properly:

```python
@app.on_event("startup")
def load_model():
    global predictor
    predictor = EnzRankService(model_path=..., csv_path=...)   # loads TensorFlow model into memory ONCE
```

Every subsequent `/rank` request reuses the already-loaded `predictor` object — much more efficient than
dGPredictor's subprocess-per-request approach. This is a good reference example if you do decide to refactor
dGPredictor to match.

### 8.3 Feature engineering (two completely different input types feeding one model)

**(a) The protein/enzyme side — sequence → integers → padding**

```python
seq_rdic = ['A','I','L','V','F','W','Y','N','C','Q','M','S','T','D','E','R','H','K','G','P','O','U','X','B','Z']
seq_dic = {w: i + 1 for i, w in enumerate(seq_rdic)}   # 'A' -> 1, 'I' -> 2, ... (0 reserved for padding)
encoded = [seq_dic[aa] for aa in protein_sequence]
padded  = pad_sequences([encoded], maxlen=2500)         # force every sequence to length 2500
```

Every amino acid letter becomes an integer 1–25; the list is truncated/zero-padded to a **fixed length of
2500** so it can be fed into a neural network (which needs fixed-size input, like an image needs fixed
pixel dimensions). This is conceptually identical to how words get turned into token IDs before being fed
into a language model, just with a 25-symbol "alphabet" (the 20 standard + a few ambiguous/rare amino acid
codes) instead of a vocabulary of words.

**(b) The substrate/chemical side — SMILES → Morgan fingerprint**

```python
mol = Chem.MolFromSmiles(smiles_string)
fp   = AllChem.GetMorganFingerprintAsBitVect(mol, useChirality=True, radius=2, nBits=2048)
```

This produces a fixed **2048-bit vector** encoding the molecule's local substructures (same family of
technique as dGPredictor's group decomposition, but here condensed into a hashed bit-vector rather than
a labeled fragment-count table — this specific technique is universally known in cheminformatics as
**ECFP4** / Morgan fingerprint radius 2). If the substrate is given as a KEGG ID instead of raw SMILES, the
code instead looks up a **precomputed** fingerprint from `CNN_data_kegg/kegg_compound.csv` (so it doesn't
need to re-derive the SMILES for well-known compounds every time).

**(c) Feeding both into the model**

```python
y = self.model.predict([comp_feature, prot_feature], verbose=0)
```

Passing a **list of two arrays** to `.predict()` is Keras syntax for a **multi-input model** — i.e. the
saved `Final_model.model` internally has two separate input branches (very likely: one branch does 1D
convolution over the padded protein sequence, another branch is a dense/MLP stack over the 2048-dim
fingerprint, and they're concatenated before a final output layer with a sigmoid activation to squash the
result into [0, 1]). This is a very standard "two-tower" neural network design for scoring compatibility
between two different types of entities (analogous to how recommendation systems score a user against
an item).

### 8.4 Important gap for you to be aware of

**The actual model-*training* script (the Keras/TensorFlow architecture definition, layers, training loop,
loss function, epochs) is NOT present anywhere in this repository.** Only the trained artifact
(`CNN_model_final/Final_model.model`, which per `missing_models_and_data.txt` is not even checked into
this git copy — see Section 13) and the *inference* code (`batch_predict.py`, loading + `.predict()`) exist
here. If you need to retrain, modify the architecture, or even just understand its exact internal layer
structure, your first move should be to load the actual `.model` file and run:

```python
loaded_model.summary()          # prints every layer, shape, and parameter count
```

...once you have the real model file (see Section 13 for how to obtain it). Treat the current CNN as a
**black box you consume**, not something to modify, until you've done that inspection.


In [ ]:

# ------------------------------------------------------------------
# TOY DEMO: reproduce EnzRank's exact FEATURE ENGINEERING (not the
# neural network itself, which we don't have) so you can see precisely
# what numbers the real CNN receives as input.
# ------------------------------------------------------------------
import numpy as np

SEQ_RDIC = ['A', 'I', 'L', 'V', 'F', 'W', 'Y', 'N', 'C', 'Q', 'M',
            'S', 'T', 'D', 'E', 'R', 'H', 'K', 'G', 'P', 'O', 'U', 'X', 'B', 'Z']
SEQ_DIC = {w: i + 1 for i, w in enumerate(SEQ_RDIC)}

def encode_and_pad(sequence, maxlen=2500):
    encoded = [SEQ_DIC[aa] for aa in sequence.upper() if aa in SEQ_DIC]
    arr = np.zeros(maxlen, dtype=int)
    # pad_sequences (Keras default) is "pre"-padding: fills zeros at the START
    if len(encoded) > maxlen:
        encoded = encoded[-maxlen:]
    arr[maxlen - len(encoded):] = encoded
    return arr

example_enzyme = "MTKRVLVTGGAGFLGSHLCERLLSEGHEVICLDNFGSGRRKNIKEFEDHPSFKVNDRDVRISESLPSVDRIYHLASRASPADFTQFPVN"
encoded_vec = encode_and_pad(example_enzyme, maxlen=2500)

print(f"Original sequence length : {len(example_enzyme)} amino acids")
print(f"Encoded feature length   : {encoded_vec.shape[0]}  (fixed, always 2500)")
print(f"Number of zero-padding   : {(encoded_vec == 0).sum()}")
print(f"First 15 encoded values  : {encoded_vec[2500-15:] }")  # last 15 = the real (non-padded) tail
print(f"(letter 'M' -> code {SEQ_DIC['M']}, 'T' -> code {SEQ_DIC['T']}, matching the sequence start)")

# --- Substrate side: Morgan fingerprint, only runs if RDKit is installed ---
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    smiles = "O[C@@H](CC([O-])=O)C([O-])=O"  # malate-like example, same as in the legacy demo app
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, useChirality=True, radius=2, nBits=2048)
    fp_array = np.array(fp)
    print(f"\nMorgan fingerprint length: {len(fp_array)} bits")
    print(f"Number of bits set to 1  : {fp_array.sum()} (a 'dense' small-molecule fingerprint is usually sparse)")
except ImportError:
    print("\n[RDKit not installed in this kernel -- skipping the fingerprint demo.")
    print(" Install with: pip install rdkit  (or conda install -c conda-forge rdkit)]")


---
## 9. Deep Dive #5 — `eQuilibrator`: The Reference-Standard Thermodynamics Service

**File:** `services/eQuilibrator/main.py` (only ~50 lines — nearly the whole service is really just a
thin FastAPI wrapper around the third-party `equilibrator_api` package).

```python
cc = ComponentContribution()          # loads a ~1.5 GB precomputed thermodynamic database into memory
cc.p_h = Q_(req.ph)
cc.p_mg = Q_(req.p_mg)
cc.ionic_strength = Q_(f"{req.ionic_strength} M")
rxn = cc.parse_reaction_formula(req.reaction_formula)   # e.g. "kegg:C00002 + kegg:C00001 <=> kegg:C00008 + kegg:C00009"
cg = cc.standard_dg_prime(rxn)
```

`equilibrator-api` implements the same family of method as dGPredictor conceptually — the *Component
Contribution* method (in fact, `dGPredictor`'s own `CC/` folder is a lightweight partial reimplementation
of ideas from the same underlying research lineage, per its `README.md`'s links). The key practical
differences that justify having **both** services:

| | eQuilibrator | dGPredictor |
|---|---|---|
| Coverage | Only compounds already in its curated database | Any molecule expressible as SMILES, including never-seen synthetic ones |
| Accuracy | Very high (peer-reviewed reference values) | Lower, comes with an explicit uncertainty (`std`) |
| Cost | ~1.5 GB database load, 1–2 min startup | Lightweight per-request subprocess |
| Use case here | "Standard" biochemical reactions | Novel pathway design |

This service takes 1–2 minutes to start up (per the README) purely because of the database load — this is
why `deploy_dev.sh` calls it out separately with a warning message.


---
## 10. Deep Dive #6 — `SearchMEResource`: Literature Pathway Lookup (No ML)

**Files:** `services/SearchMEResource/main.py`, `me_agent_logic.py`.

It's worth being explicit: **this service contains zero machine learning.** It's a well-built **data
engineering / search** component, and it's important not to mistake it for an ML tool when explaining the
project to your supervisor.

### 10.1 Data source

The underlying dataset is the supplementary material (`.xlsx` files, e.g. `41467_2025_58227_MOESM*_ESM.xlsx`)
from a published *Nature Communications* paper (the "MEResource" database referenced in the top-level
README), containing thousands of previously-computed/reported theoretical yields for producing various
target chemicals in various host-organism metabolic models.

### 10.2 Startup: build-once, cache-forever

```python
cache_path = "ya_yield_data_cache.parquet"
if os.path.exists(cache_path):
    self.yield_df = pd.read_parquet(cache_path)          # fast path (used on every restart after the first)
else:
    self._scan_excels_to_cache(cache_path)                # slow path: parses ~40 Excel files, saves parquet
```

The very first time the service starts, it globs every `.xlsx` file matching a `MOESM<N>_ESM.xlsx` naming
pattern, classifies each by which supplementary-material number range it falls in (`Base_YA`, `Hetero_YA`,
`Cofactor_YA` — different experimental categories in the paper), concatenates them all into one big
`pandas.DataFrame`, and saves that to a `.parquet` file (a compressed columnar format, much faster to
reload than re-parsing dozens of Excel files every restart — this is exactly the "1–3 minute slow cold
start" the README warns about).

### 10.3 The ranking / query logic (`query_design()`)

Given a product name, optional host organism, optional carbon source:

1. Filter rows where the target chemical **name or ID** matches (case-insensitive substring match).
2. Optionally further filter by host organism model and carbon source (also substring match).
3. Drop rows with no numeric yield value.
4. **Rank results**: sort primarily by whether the row actually lists concrete target reactions
   (`has_rxn`, so "has a route we can execute" always beats "just a yield number with no route"),
   and secondarily by the molar yield value, descending.
5. De-duplicate by the actual set of reactions (so the same pathway found in multiple source files
   doesn't appear multiple times) and return the **top 5** candidate designs.

This is a straightforward, explainable **rule-based ranking algorithm** — worth knowing precisely because
if a user ever complains "why did it suggest X and not Y", the answer is fully traceable to this sort key,
not to any opaque model.


---
## 11. Deep Dive #7 — `frontend`: The Streamlit Chat UI

**File:** `services/frontend/app.py` (a single self-contained script — this is normal for Streamlit apps;
the whole script re-runs top-to-bottom on every user interaction).

Key mechanics to understand if you'll be touching the UI:

- **`st.session_state`** is Streamlit's way of persisting Python variables across reruns (normally, every
  button click causes the entire script to execute again from line 1 — `session_state` is the escape
  hatch that survives that). This app stores **all chat sessions** (a dict keyed by a UUID) in
  `st.session_state.sessions`, so multiple parallel "conversations" can exist in the sidebar, each with its
  own `thread_id` that maps directly to the `thread_id` concept in agent-core's LangGraph checkpointer
  (Section 5.5) — i.e. the UI's "New Chat" button literally corresponds to a new LangGraph memory thread.
- **The request/response cycle**: on submit, the user's message is appended to session state, `st.rerun()`
  is called (forces Streamlit to restart the script), and *then* — because the last message's role is now
  `"user"` — a separate `if` block below fires the actual `requests.post(f"{AGENT_BASE_URL}/chat", ...)`
  call and appends the assistant's reply plus its `logs` (tool call trace) to session state. This two-phase
  "append then rerun then call" pattern is a common Streamlit idiom to get the user's own message to render
  immediately, before the (potentially slow) agent call blocks the UI.
- **Knowledge Base sidebar buttons** just read static markdown files from `services/frontend/md/` — no
  dynamic content, purely reference material shown inline.


---
## 12. Deployment & Infrastructure Mechanics

**File:** `deploy_dev.sh` (repo root).

### 12.1 Why Apptainer, not (just) Docker?

Apptainer (formerly "Singularity") is the standard container tool on shared **university/research
computing clusters**, because unlike Docker it doesn't require a persistent root-privileged daemon — a
normal user account can build and run `.sif` container images. Given this is a Final Year Project likely
deployed on a university lab server, that's almost certainly why it's used here (Dockerfiles are *also*
provided per-service, presumably for local development on a personal machine where Docker is fine).

### 12.2 What `deploy_dev.sh` actually does, step by step

```bash
BASE_DIR=$HOME/pathway_agent          # everything assumes this fixed layout on the deploy target
...
pkill -f uvicorn; pkill -f streamlit  # 1. kill any already-running instances of any service
apptainer instance stop --all         #    (idempotent restart — safe to re-run)

export APPTAINERENV_PYTHONNOUSERSITE=1
export APPTAINERENV_PYTHONUNBUFFERED=1   # forces Python's print() to flush immediately (so logs stream live)

nohup apptainer exec \
  --bind $CODE_DIR/dGPredictor:/app \    # 2. mounts the LIVE code folder INTO the container at /app
  $IMG_DIR/dg.sif \
  bash -c "cd /app && uvicorn server:app --host 0.0.0.0 --port 8002" > $LOG_DIR/dg.log 2>&1 &
                                          #    runs it in the background, redirecting stdout+stderr to a log file
```

Repeated (with slightly different flags) for all 6 services. Two details worth internalizing:

1. **`--bind $CODE_DIR/<service>:/app`** — the container image itself only provides the *Python
   environment* (installed libraries); the actual application code is bind-mounted live from the host
   filesystem. This means **editing a `.py` file and restarting the relevant service is enough** — you do
   **not** need to rebuild the `.sif` image just to change application logic, only when you change
   dependencies (`requirements.txt` / the `.def` file).
2. **Environment variables wire the services together**, e.g.:
   ```bash
   export APPTAINERENV_DG_PREDICTOR_URL=http://127.0.0.1:8002/predict
   ```
   The `APPTAINERENV_` prefix is Apptainer's convention for "pass this environment variable through into
   the container". Inside the container, `agent-core`'s `app/config.py` reads it back out via
   `os.getenv("DG_PREDICTOR_URL", "http://127.0.0.1:8002/predict")`. **If you ever change a port, you must
   update it in both `deploy_dev.sh` and `config.py`'s matching env var/default** (the README explicitly
   warns about this — a very likely source of "why can't agent-core reach my tool" bugs).
3. **Ordering matters but is soft**: tool services (dG, EnzRank, SearchME, eQuilibrator) start first, then
   after a short `sleep 3`, agent-core and frontend start last, since they depend on the tool services
   (and Ollama) already being reachable.
4. **Persistence bind mount**: `--bind $DATA_DIR:/persistence` maps the host's
   `~/pathway_agent/data/agent_memory` folder into the container so the SQLite checkpoint database survives
   container restarts (otherwise it would live inside the ephemeral container filesystem and be lost).

### 12.3 Local LLM serving via Ollama

Ollama is a separate program (not part of this codebase) that you install once on the host machine. It
downloads and serves open-weight LLMs (here `qwen2.5:7b`) over a local HTTP API on port 11434. Running the
LLM locally (instead of calling OpenAI/Anthropic's cloud API) means: no per-token cost, no data leaving the
server (relevant for possibly-sensitive unpublished research), but a **weaker model** than top cloud LLMs —
which is exactly why Section 5.4's very explicit system prompt and JSON-parsing fallback exist: to
compensate for a smaller model's lower reliability at following instructions and using tools correctly.


---
## 13. ⚠️ What This Git Copy Is Missing (read before you try to run anything)

**File:** `missing_models_and_data.txt` (repo root) — a list of **~10,700 file paths** that exist on the
original deployment but were intentionally excluded from this git repository (they're either too large for
git, or environment-specific build artifacts).

The README says this explicitly:

> *"Do not use git clone alone, as large model weights and local databases are excluded from Git. Download
> the full asset package [from the Releases page]."*

Categories of what's missing, grouped from that file:

| Category | Examples | Why it's excluded from git |
|---|---|---|
| **Trained model weights** | `CNN_model_final/Final_model.model` (EnzRank), `model/M12_model_BR.pkl` (dGPredictor) | Binary model files, can be large; also environment/build-generated |
| **Built container images** | `images/*.sif` (all 6+ services), plus an entire unpacked `images/search_me_sandbox/` directory tree | `.sif` files are large binary build artifacts, meant to be built locally via `apptainer build`, not committed |
| **Runtime state** | `data/agent_memory/agent_memory.sqlite(-shm/-wal)` | This is generated fresh on each deployment (it's the conversation memory database, not source code) |
| **Large reference databases** | eQuilibrator's ~1.5 GB thermodynamic database, KEGG cached data | Third-party scientific databases, not source code, and too large for git |
| **The literature yield dataset** | `Dataset/*.xlsx` (~40 files) + the generated `ya_yield_data_cache.parquet` for SearchMEResource | Supplementary material from a published paper — large binary spreadsheets |
| **Misc redis/dump artifacts** | `images/dump.rdb`, `images/redis.sif` | Suggests a Redis cache may be used in some deployment configuration not fully reflected in the services shown here — worth asking the previous author about, since no `redis` usage is visible in the current Python source of any service |

### 13.1 Practical implication for you

**You cannot run this system end-to-end from a fresh `git clone` alone.** To actually run it, you need the
full release tarball (`pathway_agent_full_vX.X.tar.gz`) from the project's GitHub Releases page, which
bundles all of the above. Cloning git only gets you the **source code** — this notebook, and reading the
code directly, is how you understand the *logic*; the release tarball is how you get a *runnable* system.

If your task is to modify code (not retrain models), you generally don't need the model weights at all for
services you're not touching — but you do need at least the specific model/data files for whichever
service you're actively working on, plus Ollama + a pulled model, to test end-to-end.

Also worth flagging in your report to your supervisor: the presence of `redis.sif`/`dump.rdb` in the
missing-files list but with **no corresponding Redis client code anywhere in the current `services/`
Python source** suggests either (a) a leftover from an earlier architecture that was later removed, or
(b) a caching layer that exists on the deployment server but whose integration code isn't checked into this
repo — worth a quick question to whoever maintained the original deployment, rather than assuming.


---
## 14. Evaluation & Testing Framework

**Files:** `services/agent-core/evaluation/{generate_dataset.py, run_eval.py, score_eval.py,
visualize_report.py, test_dataset_rich.json, run_tests.sh, run_scoring.sh}`.

This is likely central to the FYP's academic evaluation/reporting requirements, so it's worth understanding
well even if you're not asked to touch it directly.

### 14.1 Two-stage pipeline

**Stage 1 — `run_tests.sh` → `run_eval.py`: generate raw transcripts**

- Loads a fixed set of test questions from `test_dataset_rich.json` (each with an `id`, a `query`, a
  `category`, and an `expected_tool`/`expected_tools` — the "correct" tool(s) a good agent should call).
- For **each** test case: wipes the SQLite agent-memory database clean (`clear_agent_memory()`) so test
  cases can't leak state into each other, then calls `app_graph.invoke(...)` **directly in-process**
  (importing `app_graph` from `app.agents.graph` — bypassing the FastAPI/HTTP layer entirely, which makes
  this faster and removes network variables from the measurement).
- Records: latency (seconds), which tools were *actually* called vs. *expected*, the final raw text answer,
  and any crash/error.
- Saves everything to a timestamped CSV (`reports/eval_report_<timestamp>.csv`).

**Stage 2 — `run_scoring.sh` → `score_eval.py`: LLM-as-a-judge grading**

- Finds the most recent `eval_report_*.csv`.
- For every row, sends a **new prompt to a (potentially different) LLM** (`llama3.1` is hard-coded as the
  judge model here, separate from the `qwen2.5:7b` agent model) asking it to grade the transcript on:
  **Accuracy** (1–5), **Reasoning** (1–5), **Completeness** (1–5), producing a `total_score` (max 15) and
  a free-text `improvement_suggestion`.
- Saves an augmented "scored" CSV.

This is the **"LLM-as-a-judge"** pattern from the glossary in action — a common, pragmatic way to evaluate
open-ended agent behavior at scale without needing a human to manually read every transcript. It's
important to communicate this clearly to a boss/supervisor: these scores are an **automated proxy metric**
using a second LLM's judgment, not ground-truth/human-verified accuracy — a reasonable and standard
practice in current agent research, but a limitation worth stating explicitly in any report.

### 14.2 `visualize_report.py`

Turns the scored CSV into charts (likely score distributions / per-category breakdowns) — check this file
directly when preparing a results section for your supervisor, since chart types/labels there will match
whatever the FYP's report expects.


In [ ]:

# Peek at the actual evaluation test-case categories and expected tools shipped with the repo,
# to see exactly what capabilities the agent is being tested against.
import json
from pathlib import Path

candidates = [
    Path.cwd() / "services" / "agent-core" / "evaluation" / "test_dataset_rich.json",
    Path("/Users/yiranzhu/Desktop/python/FYP2026xintong-main/services/agent-core/evaluation/test_dataset_rich.json"),
]

path = next((p for p in candidates if p.exists()), None)

if path:
    data = json.loads(path.read_text(encoding="utf-8"))
    print(f"Loaded {len(data)} test cases from {path}\n")

    import pandas as pd
    df = pd.DataFrame(data)
    if "category" in df.columns:
        print("Test cases per category:")
        print(df["category"].value_counts().to_string())
    print("\nFirst 3 example test cases:")
    for case in data[:3]:
        print(f"  id={case.get('id')!r} | category={case.get('category')!r}")
        print(f"    query: {case.get('query')}")
        print(f"    expected tool(s): {case.get('expected_tool') or case.get('expected_tools')}\n")
else:
    print("test_dataset_rich.json not found at the expected path(s) -- adjust the path above.")


---
## 15. End-to-End Trace of One Real Example

Let's trace the exact sample query from the README, hop by hop, referencing exact functions:

> **User types:** *"Calculate the Gibbs free energy for the reaction: C01083 + C00001 <=> 2 C00031"*

1. **`frontend/app.py`** — appends the message to `session_state`, `st.rerun()`s, then POSTs
   `{"query": "...", "thread_id": "<session-uuid>"}` to `http://agent-core:8081/chat`.
2. **`agent-core/main.py`** `chat()` — calls `run_graph(query, thread_id)`.
3. **`graph.py`** `run_graph()` → `app_graph.invoke({"messages": [HumanMessage(...)]}, config={"thread_id": ...})`.
4. **Graph enters the `"agent"` node** (`chatbot()`): system prompt (Section 5.4) + message get sent to
   Ollama's `qwen2.5:7b`. Per the system prompt's **Scenario 1** ("User provides IDs"), the model's first
   move should be to call `entity_validation_tool` for `C01083`, `C00001`, and `C00031` to confirm their
   identities (mandatory decoding step).
5. Each validation call routes: `"agent"` → `should_continue` sees `tool_calls` populated → **`"tools"`
   node** → `entity_validation_tool()` in `db_search.py` hits `https://rest.kegg.jp/get/cpd/C01083` etc.,
   parses out the `NAME`/`FORMULA` lines, returns a formatted string → back to **`"agent"`** node with the
   result appended as a `ToolMessage`.
6. Once IDs are confirmed, per the system prompt's thermodynamics rule, since these are **known KEGG
   compounds** the model should prefer `equilibrator_tool` (not `dg_predictor_tool`) — and must add the
   `kegg:` prefix. It calls `equilibrator_tool` with
   `"kegg:C01083 + kegg:C00001 <=> 2 kegg:C00031"`.
7. **`remote_equilibrator.py`** regex-normalizes the prefix (belt-and-braces even though the LLM was
   already told to add it) and POSTs to `http://eQuilibrator:8005/calculate`.
8. **`eQuilibrator/main.py`** parses the formula with `equilibrator_api`, computes `standard_dg_prime()`
   at the given pH/pMg/ionic strength, returns `{"dG_prime_molar": "...", "dG_error": "...", ...}`.
9. Result flows back as a `ToolMessage` → **`"agent"`** node again → the LLM now has everything it needs
   and (per the "CRITICAL FORMATTING RULE") writes a **plain-text, non-LaTeX** final answer like
   *"dG = -34.5 kJ/mol (±1.2), this reaction is thermodynamically favorable."*
10. `should_continue` sees no `tool_calls` this time → routes to **`END`**.
11. `run_graph()` slices out all the new messages, builds the human-readable `logs` list (one entry per
    tool call + one entry per tool result, truncated to 200 characters), and returns
    `(final_text, logs)`.
12. **`main.py`** `chat()` wraps that into `{"response": ..., "logs": [...], "thread_id": ...}` and returns
    it as the HTTP response.
13. **`frontend/app.py`** renders `final_res` as the chat bubble, and `logs` inside the collapsible
    "View Tool Execution Logs (Internal)" expander.

This whole round trip is a **single POST /chat request** from the frontend's point of view — all the
back-and-forth in steps 4–10 happens synchronously inside `agent-core` before it ever responds.


---
## 16. Suggested Onboarding Path (as a new engineer)

Given you're a beginner in ML but presumably comfortable with software engineering, here's a sensible order
to actually build competence, mapped to what you now know from this notebook:

### Step 1 — Read code with this notebook open side-by-side
1. `services/agent-core/app/agents/graph.py` (Section 5) — the most important file in the repo.
2. `services/agent-core/app/tools/*.py` (Section 6).
3. `services/agent-core/app/config.py` — trivial but tells you every wiring point.

### Step 2 — Get *something* running locally, incrementally
You don't need the full stack to start being productive. A safe path:
1. Get **Ollama** running locally and `ollama pull qwen2.5:7b` (or a smaller model if your machine is
   limited — you can point `Config.MODEL_NAME` at anything Ollama supports, e.g. a smaller `qwen2.5:1.5b`,
   while developing, then switch back).
2. Run `agent-core` **without** any of the 4 tool services up — you'll be able to chat with the raw LLM +
   see it *attempt* tool calls that fail with the friendly `"Connection Error: ..."` messages from
   `base.py`'s `post_request()`. This alone lets you iterate on the system prompt (Section 5.4) safely.
3. Bring up **one** tool service at a time (e.g. `SearchMEResource`, since it needs no GPU and no huge
   model file — just the Excel dataset) and confirm end-to-end tool calling works for that one tool.
4. Only pull in `dGPredictor` / `EnzRank` / `eQuilibrator` (and their large model/data files, Section 13)
   once you specifically need to test or modify them.

### Step 3 — Safe, well-scoped first contributions
Some concrete, low-risk tasks that would demonstrate real understanding without requiring you to touch the
ML models themselves:
- Convert `dGPredictor`'s subprocess-per-request pattern to a load-once-at-startup pattern like `EnzRank`
  (Section 7.1) — a clear performance win, fully testable by comparing response latency before/after.
- Add automated (non-LLM-judged) **unit tests** for the pure-Python logic that has no ML dependency at all:
  `_validate_equation()` in `remote_dg.py`, `preprocess_kegg_query()` in `db_search.py`,
  `parse_reaction_formula_side()` / `parse_formula()` in `db_bulk_dg_gen.py`. These are deterministic
  functions — perfect first unit-test targets.
- Investigate and document (or remove) the `redis.sif`/`dump.rdb` mismatch noted in Section 13.1.
- Extend `test_dataset_rich.json` with new evaluation cases for behaviors you notice are under-tested.

### Step 4 — Once comfortable, go deeper into the ML internals
- Load the real EnzRank `.model` file and run `.summary()` to finally see its true architecture
  (Section 8.4) — document what you find, since currently no one file in the repo describes it.
- Read `services/dGPredictor/CC/` in full (thermodynamic_constants.py, compound.py, compound_cacher.py) to
  understand the pH-correction math referenced in Section 7.2 in more depth.

### What to tell your boss, in one paragraph
*"This is a multi-service AI agent for synthetic-biology pathway design. A local open-weight LLM
(orchestrated with LangGraph as a ReAct-style tool-using agent) answers natural-language questions by
calling four specialist backends: a classical machine-learning model (Bayesian Ridge Regression over
RDKit-derived molecular fragments) for novel-reaction thermodynamics, a convolutional neural network for
enzyme-substrate compatibility scoring, a third-party reference thermodynamics library for known reactions,
and a curated literature database search for known pathway yields — plus direct calls to public
bioinformatics APIs (KEGG/Rhea/UniProt) for identifier lookup. The system is containerized with Apptainer
for deployment on shared research infrastructure, and includes an automated LLM-as-a-judge evaluation
harness to score agent quality. The current git repository holds only source code; large trained models,
reference databases, and built container images (~10,700 files) must be obtained separately via the
project's release package."*


---
## 17. Appendix — Config & Ports Cheat Sheet

| Env var (in `deploy_dev.sh`) | Read by (`config.py` key) | Default value |
|---|---|---|
| `DG_PREDICTOR_URL` | `Config.DG_PREDICTOR_URL` | `http://127.0.0.1:8002/predict` |
| `ENZRANK_URL` | `Config.ENZRANK_URL` | `http://127.0.0.1:8003/rank` |
| `MERESOURCE_URL` | `Config.MERESOURCE_URL` | `http://127.0.0.1:8004` |
| `EQUILIBRATOR_URL` | `Config.EQUILIBRATOR_URL` | `http://127.0.0.1:8005/calculate` |
| `OLLAMA_HOST` → `OLLAMA_BASE_URL` | `Config.OLLAMA_BASE_URL` | `http://127.0.0.1:11434` |
| `AGENT_MEMORY_PATH` | read directly via `os.getenv` in `graph.py` | `data/agent_memory/agent_memory.sqlite` |
| *(hardcoded, not env)* | `Config.MODEL_NAME` | `"qwen2.5:7b"` — change this + re-`ollama pull` to switch LLMs |
| *(hardcoded, not env)* | `Config.KEGG_API_URL` / `RHEA_API_URL` / `UNIPROT_API_URL` | public REST endpoints, no auth needed |

### Quick reference: which service is ML, and what kind

| Service | ML? | Algorithm family |
|---|---|---|
| agent-core | Yes (the LLM itself) | Pretrained transformer LLM (`qwen2.5:7b`), used via prompting + tool-calling — **not trained/fine-tuned by this project** |
| dGPredictor | Yes | Bayesian Ridge Regression (classic/linear ML) over hand-engineered RDKit fragment features |
| EnzRank | Yes | Convolutional Neural Network (deep learning), dual-input (sequence + fingerprint) |
| eQuilibrator | No (uses precomputed reference values + physics/statistics formulas from an external, non-learned method) | — |
| SearchMEResource | No | Rule-based filtering + sorting over a static dataset |
| frontend | No | Plain UI code |

---

*End of notebook. This file was generated to help onboard a new engineer without altering any original
project files — safe to keep, edit, and re-run indefinitely.*
